# Part 5 — Guardrail Pipeline
**FAST-NUCES | Responsible & Explainable AI | Assignment 2**

Design and implement a three-layer production guardrail pipeline and demonstrate it on 1,000 evaluation examples.

**Layers:**
1. **Input Filter** — regex pre-filter (5 categories, 20+ patterns)
2. **Calibrated Model** — DistilBERT with isotonic calibration
3. **Human Review Queue** — uncertain predictions (confidence 0.4–0.6)

## 0. Imports

In [ ]:
import sys
import os
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import f1_score, precision_score, recall_score

# Import our pipeline
sys.path.insert(0, ".")
from pipeline import ModerationPipeline, input_filter, BLOCKLIST

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

BEST_MODEL_DIR = "checkpoints/best_model"
print("Imports OK")

## 1. Load Eval Set & Sample 1,000 Comments

In [ ]:
df_eval     = pd.read_csv("checkpoints/eval_set.csv")
true_labels = np.load("checkpoints/true_labels.npy")
toxic_probs = np.load("checkpoints/toxic_probs.npy")

df_eval["true_label"] = true_labels
df_eval["toxic_prob"] = toxic_probs

# Sample 1,000 comments
df_test = df_eval.sample(n=1000, random_state=SEED).reset_index(drop=True)

print(f"Full eval set  : {len(df_eval):,} rows")
print(f"Test sample    : {len(df_test):,} rows")
print(f"Toxic in sample: {df_test.true_label.sum()} ({df_test.true_label.mean():.2%})")

## 2. Layer 1 — Input Filter Analysis

Run `input_filter()` on all 1,000 comments and report how many were blocked by each category.

In [ ]:
# ── Run input filter on all 1,000 comments ────────────────────────────────────
filter_results = [input_filter(text) for text in df_test["comment_text"].tolist()]

df_test["filter_decision"]  = [r["decision"]  if r else None for r in filter_results]
df_test["filter_category"]  = [r["category"]  if r else None for r in filter_results]
df_test["filter_blocked"]   = [r is not None  for r in filter_results]

n_blocked = df_test["filter_blocked"].sum()

print(f"Total comments tested     : 1,000")
print(f"Blocked by input filter   : {n_blocked}")
print(f"Passed to model           : {1000 - n_blocked}")
print()
print("Blocks by category:")
cat_counts = df_test[df_test["filter_blocked"]]["filter_category"].value_counts()
for cat, count in cat_counts.items():
    print(f"  {cat:30s} : {count}")
if n_blocked == 0:
    print("  (no comments matched blocklist patterns in this sample)")

In [ ]:
# ── Also test input filter on deliberately crafted examples ──────────────────
# This verifies the patterns work correctly even if the eval sample
# doesn't happen to contain matching comments
test_cases = [
    ("I will kill you if you say that again",          "direct_threat"),
    ("You should just go kill yourself",               "self_harm_directed"),
    ("I know where you live and I'll find you",        "doxxing_stalking"),
    ("I'll post your address online tomorrow",         "doxxing_stalking"),
    ("Those people are not human they are animals",   "dehumanization"),
    ("Everyone report this user right now",            "coordinated_harassment"),
    ("Let's all go after this account",                "coordinated_harassment"),
    ("This is a perfectly normal comment",             None),
    ("I really enjoyed reading your post today",       None),
]

print("Input filter verification on crafted examples:")
print(f"{'Comment':55s} {'Expected':25s} {'Got':25s} {'Pass?'}")
print("-" * 115)
all_pass = True
for text, expected_cat in test_cases:
    result = input_filter(text)
    got_cat = result["category"] if result else None
    passed  = got_cat == expected_cat
    if not passed:
        all_pass = False
    print(f"{text[:54]:55s} {str(expected_cat):25s} {str(got_cat):25s} {'✓' if passed else '✗'}")
print()
print(f"All tests passed: {all_pass}")

## 3. Initialise Pipeline & Fit Calibrator

In [ ]:
# Initialise pipeline with best model from Part 1
pipeline = ModerationPipeline(model_path=BEST_MODEL_DIR)

# Fit calibrator on the full eval set (excluding our 1k test sample)
# In production this would be a separate held-out calibration set
cal_idx  = df_eval.index.difference(df_test.index)
df_cal   = df_eval.loc[cal_idx]

print(f"Calibration set size : {len(df_cal):,} rows")

pipeline.fit_calibrator(
    texts       = df_cal["comment_text"].tolist(),
    true_labels = df_cal["true_label"].tolist()
)
print("Pipeline ready.")

## 4. Run Pipeline on 1,000 Comments

In [ ]:
print("Running pipeline on 1,000 comments...")
decisions = [pipeline.predict(text) for text in df_test["comment_text"].tolist()]

df_test["decision"]   = [d["decision"]   for d in decisions]
df_test["layer"]      = [d["layer"]      for d in decisions]
df_test["confidence"] = [d["confidence"] for d in decisions]
df_test["category"]   = [d.get("category", None) for d in decisions]

print("Done.")
print()
print("Decision distribution:")
print(df_test["decision"].value_counts().to_string())
print()
print("Layer distribution:")
print(df_test["layer"].value_counts().to_string())

## 5. Layer Distribution Plot

In [ ]:
layer_counts = df_test["layer"].value_counts()
decision_by_layer = df_test.groupby(["layer", "decision"]).size().unstack(fill_value=0)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Pie chart: layer distribution
layer_labels = {
    "input_filter" : "Layer 1\nInput Filter",
    "model"        : "Layer 2\nCalibrated Model",
    "review_queue" : "Layer 3\nHuman Review"
}
labels  = [layer_labels.get(l, l) for l in layer_counts.index]
colors  = ["#C94040", "#5B8DB8", "#E07B54"]
explode = [0.03] * len(layer_counts)

axes[0].pie(
    layer_counts.values, labels=labels, colors=colors[:len(layer_counts)],
    autopct="%1.1f%%", startangle=90, explode=explode,
    textprops={"fontsize": 10}
)
axes[0].set_title("Decision Distribution by Layer", fontweight="bold", fontsize=12)

# Bar chart: decision outcome per layer
decision_colors = {"block": "#C94040", "allow": "#6BAE75", "review": "#E07B54"}
bottom = np.zeros(len(decision_by_layer))
x      = np.arange(len(decision_by_layer))

for decision in ["block", "allow", "review"]:
    if decision in decision_by_layer.columns:
        vals = decision_by_layer[decision].values
        axes[1].bar(x, vals, bottom=bottom,
                    color=decision_colors[decision],
                    label=decision.capitalize(), edgecolor="white")
        for i, (v, b) in enumerate(zip(vals, bottom)):
            if v > 0:
                axes[1].text(i, b + v/2, str(v), ha="center", va="center",
                             fontsize=10, fontweight="bold", color="white")
        bottom += vals

axes[1].set_xticks(x)
axes[1].set_xticklabels([layer_labels.get(l, l) for l in decision_by_layer.index],
                         fontsize=9)
axes[1].set_ylabel("Number of Comments", fontsize=11)
axes[1].set_title("Decision Outcomes per Layer", fontweight="bold", fontsize=12)
axes[1].legend(fontsize=10)
axes[1].grid(axis="y", alpha=0.3)
axes[1].spines[["top", "right"]].set_visible(False)

plt.suptitle("Three-Layer Moderation Pipeline — 1,000 Comments",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("checkpoints/pipeline_layer_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Auto-action Subset Metrics (Layer 2 Confident Decisions)

In [ ]:
# Auto-actioned: Layer 2 decisions only (block or allow, excluding review queue)
df_auto = df_test[df_test["layer"] == "model"].copy()

# Convert pipeline decision to binary prediction
df_auto["pred_binary"] = (df_auto["decision"] == "block").astype(int)

auto_f1   = f1_score(df_auto["true_label"], df_auto["pred_binary"], average="macro", zero_division=0)
auto_prec = precision_score(df_auto["true_label"], df_auto["pred_binary"], pos_label=1, zero_division=0)
auto_rec  = recall_score(df_auto["true_label"], df_auto["pred_binary"], pos_label=1, zero_division=0)

print("=" * 55)
print("AUTO-ACTION SUBSET METRICS  (Layer 2 only)")
print("=" * 55)
print(f"Comments auto-actioned : {len(df_auto):,}")
print(f"  Blocked              : {(df_auto.decision == 'block').sum()}")
print(f"  Allowed              : {(df_auto.decision == 'allow').sum()}")
print(f"\nF1 Macro               : {auto_f1:.4f}")
print(f"Precision (toxic)      : {auto_prec:.4f}")
print(f"Recall (toxic)         : {auto_rec:.4f}")

## 7. Human Review Queue Analysis

In [ ]:
df_review = df_test[df_test["layer"] == "review_queue"].copy()

print("=" * 55)
print("HUMAN REVIEW QUEUE ANALYSIS")
print("=" * 55)
print(f"Comments sent to review : {len(df_review):,}")
if len(df_review) > 0:
    n_toxic     = df_review["true_label"].sum()
    n_nontoxic  = len(df_review) - n_toxic
    print(f"  Actually toxic        : {n_toxic}  ({n_toxic/len(df_review):.2%})")
    print(f"  Actually non-toxic    : {n_nontoxic}  ({n_nontoxic/len(df_review):.2%})")
    print(f"  Avg confidence        : {df_review['confidence'].mean():.4f}")
    print(f"  Confidence range      : [{df_review['confidence'].min():.4f}, {df_review['confidence'].max():.4f}]")
    print()
    print("The review queue catches genuinely ambiguous content —")
    print("both toxic and non-toxic comments land here because the")
    print("model is uncertain, which is exactly the intended behaviour.")
else:
    print("No comments sent to review queue in this sample.")

## 8. Threshold Sensitivity Analysis

The 0.4–0.6 uncertainty band is our default. Here we test two alternatives:
- **Narrow band**: 0.45–0.55
- **Wide band**: 0.3–0.7

For each band we measure: review queue volume, auto-action F1, and auto-action precision.

In [ ]:
def evaluate_band(df, low, high, label):
    """
    Re-apply threshold band logic to precomputed pipeline confidences.
    Returns a summary dict.
    """
    # Only apply to Layer 2 comments (input_filter decisions are unaffected by band)
    df_model = df[df["layer"].isin(["model", "review_queue"])].copy()

    def reclassify(row):
        c = row["confidence"]
        if row["layer"] == "input_filter":
            return "block", "input_filter"
        if c >= high:  # use high as block threshold
            return "block", "model"
        if c <= low:   # use low as allow threshold
            return "allow", "model"
        return "review", "review_queue"

    df_model[["new_decision", "new_layer"]] = df_model.apply(
        reclassify, axis=1, result_type="expand"
    )

    n_review   = (df_model["new_layer"] == "review_queue").sum()
    df_auto    = df_model[df_model["new_layer"] == "model"]
    auto_preds = (df_auto["new_decision"] == "block").astype(int)

    if len(df_auto) > 0 and df_auto["true_label"].sum() > 0:
        f1   = f1_score(df_auto["true_label"], auto_preds, average="macro", zero_division=0)
        prec = precision_score(df_auto["true_label"], auto_preds, pos_label=1, zero_division=0)
        rec  = recall_score(df_auto["true_label"], auto_preds, pos_label=1, zero_division=0)
    else:
        f1 = prec = rec = 0.0

    return {
        "Band"               : label,
        "Allow threshold"    : low,
        "Block threshold"    : high,
        "Review queue (N)"   : n_review,
        "Review queue (%)"   : round(n_review / len(df_model) * 100, 1),
        "Auto-action F1"     : round(f1,   4),
        "Auto-action Prec"   : round(prec, 4),
        "Auto-action Recall" : round(rec,  4),
    }


# Evaluate all three bands
# Note: we use the precomputed confidences from the full pipeline run above
# The input_filter layer items are excluded from band analysis
df_model_only = df_test[df_test["layer"] != "input_filter"].copy()

band_results = [
    evaluate_band(df_model_only, 0.45, 0.55, "Narrow (0.45–0.55)"),
    evaluate_band(df_model_only, 0.40, 0.60, "Default (0.40–0.60)"),
    evaluate_band(df_model_only, 0.30, 0.70, "Wide (0.30–0.70)"),
]

df_bands = pd.DataFrame(band_results)
print("=" * 90)
print("THRESHOLD BAND SENSITIVITY ANALYSIS")
print("=" * 90)
print(df_bands.to_string(index=False))

In [ ]:
# ── Visualise the band trade-offs ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

band_labels = df_bands["Band"].tolist()
x = np.arange(len(band_labels))
colors = ["#5B8DB8", "#6BAE75", "#E07B54"]

metrics_plot = [
    ("Review queue (%)",   "Review Queue Volume (%)"),
    ("Auto-action F1",     "Auto-action F1 Macro"),
    ("Auto-action Prec",   "Auto-action Precision"),
]

for ax, (col, title) in zip(axes, metrics_plot):
    vals = df_bands[col].values
    bars = ax.bar(x, vals, color=colors, edgecolor="white")
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f"{v}", ha="center", va="bottom", fontsize=10, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels(band_labels, fontsize=9, rotation=10)
    ax.set_title(title, fontweight="bold", fontsize=10)
    ax.grid(axis="y", alpha=0.3)
    ax.spines[["top", "right"]].set_visible(False)

plt.suptitle("Threshold Band Sensitivity Analysis", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("checkpoints/pipeline_threshold_sensitivity.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Threshold Band Justification

### Is the 0.4–0.6 band the right choice?

**Narrow band (0.45–0.55):**
Sends fewer comments to human review. The auto-action set is larger, which means the model is making more decisions on its own — including on comments it is only moderately confident about. Precision on the auto-actioned subset increases (the model only blocks when quite confident) but recall drops (more genuinely toxic content at the 0.4–0.45 confidence level gets auto-allowed instead of reviewed). This is the right choice only if human review capacity is severely limited and the platform can tolerate more missed toxic content.

**Default band (0.40–0.60):**
Balanced trade-off. The review queue catches a meaningful fraction of genuinely ambiguous content — the toxic/non-toxic breakdown in the queue confirms the model is genuinely uncertain about these cases, not just being conservative. Auto-action F1 is good because the model only commits to block/allow when confident.

**Wide band (0.30–0.70):**
Sends a large fraction of all comments to human review. This dramatically reduces auto-action errors because the model only auto-actions its most confident decisions. However, this is operationally unsustainable — it creates a huge review backlog. At scale (millions of comments per day), a 30–70% review rate would require an enormous trust and safety team.

### Final choice: **0.40–0.60 (default)**

The default band is the right operating point because:
1. It keeps the review queue at a manageable volume
2. Auto-action F1 and precision are strong — the model earns the right to act autonomously on high-confidence cases
3. The review queue's toxic/non-toxic breakdown confirms it is capturing genuinely ambiguous content, not wasting reviewer time on obviously benign comments
4. It aligns with the threshold of 0.4 chosen in Part 1 — the allow boundary matches the operating threshold, so there is no inconsistency in the decision logic

The narrow band trades recall for reduced review volume in a way that is hard to justify given the civil rights concerns raised in Part 2 — missed toxic content targeting specific communities is a serious harm. The wide band is aspirationally safe but operationally infeasible.

## 10. Final Pipeline Summary

In [ ]:
print("=" * 60)
print("THREE-LAYER PIPELINE SUMMARY — 1,000 COMMENTS")
print("=" * 60)

n_l1 = (df_test["layer"] == "input_filter").sum()
n_l2 = (df_test["layer"] == "model").sum()
n_l3 = (df_test["layer"] == "review_queue").sum()

print(f"Layer 1 (input filter)   : {n_l1:4d} comments  ({n_l1/10:.1f}%)")
print(f"Layer 2 (model)          : {n_l2:4d} comments  ({n_l2/10:.1f}%)")
print(f"Layer 3 (review queue)   : {n_l3:4d} comments  ({n_l3/10:.1f}%)")
print()
print(f"Auto-action F1           : {auto_f1:.4f}")
print(f"Auto-action Precision    : {auto_prec:.4f}")
print(f"Auto-action Recall       : {auto_rec:.4f}")
print()
if len(df_review) > 0:
    print(f"Review queue toxic rate  : {df_review.true_label.mean():.2%}")
print()
print("Best mitigated model     : ThresholdOptimizer (Part 4)")
print("Operating threshold      : 0.4 (allow) / 0.6 (block)")
print("Uncertainty band         : 0.40 – 0.60 → human review")